FACET Tutorial 1 Equivalent: Classification with GAMMA_DNA
Mirrors Classification_with_Facet.ipynb — Titanic survival prediction.

Covers:
  - EDA (g.skim, g.quality, g.eda)
  - Training RandomForestClassifier
  - SHAP feature importance (g.explain)
  - plot_overview, plot_synergy, plot_redundancy (kind="both")
  - top_interactions
  - plot_pdp
  - simulate

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.io as pio

print("=" * 60)
print("FACET Tutorial 1: Titanic Classification")
print("=" * 60)

FACET Tutorial 1: Titanic Classification


## 1. Load & preprocess Titanic

In [3]:
import seaborn as sns
df_raw = sns.load_dataset("titanic")
print(f"Raw shape: {df_raw.shape}")

COLS = ["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "who"]
df = df_raw[COLS].copy()

df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna("S")

df["sex_enc"]      = (df["sex"] == "female").astype(int)
df["embarked_enc"] = df["embarked"].map({"S": 0, "C": 1, "Q": 2}).fillna(0).astype(int)
df["who_enc"]      = df["who"].map({"man": 0, "woman": 1, "child": 2}).fillna(0).astype(int)

FEATURES = ["pclass", "sex_enc", "age", "sibsp", "parch", "fare", "embarked_enc", "who_enc"]
MODEL_DF = df[["survived"] + FEATURES].copy()
print(f"Model-ready shape: {MODEL_DF.shape}, nulls: {MODEL_DF.isnull().sum().sum()}")

Raw shape: (891, 15)
Model-ready shape: (891, 9), nulls: 0


## 2. GAMMA_DNA session

In [4]:
from gamma.pipeline import GAMMA_DNA

g = GAMMA_DNA(MODEL_DF, target="survived", name="facet_titanic")
print("\n[1] Session created")
print(g)


[1] Session created
GAMMA_DNA(name='facet_titanic', target='survived', task='binary_classification', frames=['raw'], experiments=0)


## 3. EDA

In [5]:
print("\n[2] Quick data overview (skim):")
try:
    g.skim()
    print("  ✓ skim() OK")
except Exception as e:
    print(f"  ✗ skim(): {e}")

print("\n[3] Data quality:")
try:
    g.quality()
    print("  ✓ quality() OK")
except Exception as e:
    print(f"  ✗ quality(): {e}")


[2] Quick data overview (skim):
  ✓ skim() OK

[3] Data quality:
  ✓ quality() OK


## 4. Train

In [6]:
print("\n[4] Training RandomForestClassifier:")
result = g.train(
    model_type="random_forest_classifier",
    features=FEATURES,
    test_size=0.2,
    random_state=42,
    run_cv=True,
    cv_folds=5,
    model_params={"n_estimators": 100, "max_depth": 6, "random_state": 42},
)
print(f"  ✓ train() OK  |  test AUC = {getattr(result, 'roc_auc', 'n/a')}")


[4] Training RandomForestClassifier:
  ✓ train() OK  |  test AUC = 0.8469696969696969


## 5. Explain

In [7]:
print("\n[5] Feature importance (SHAP):")
report = g.explain(compute_shap=True, compute_permutation=False)
print(f"  ✓ explain() OK  |  {report}")


[5] Feature importance (SHAP):
  Computing SHAP values on 8 features… (may take 10-60s)
  Done.
  ✓ explain() OK  |  FeatureImportanceReport(features=8, model_imp=yes, perm_imp=no, shap=yes)


## 6. XAI: Overview

In [8]:
print("\n[6] plot_overview:")
fig = report.plot_overview(max_display=8)
assert len(fig.data) >= 2, "overview should have >= 2 traces"
print(f"  ✓ plot_overview OK  |  traces={len(fig.data)}")


[6] plot_overview:


  ✓ plot_overview OK  |  traces=9


In [12]:
report.plot_facet()

## 7. XAI: Synergy

In [13]:
print("\n[7] Synergy analysis (interaction SHAP):")
fig_syn = report.plot_synergy(kind="both", sample_size=300)
heatmaps = [t for t in fig_syn.data if t.type == "heatmap"]
scatters  = [t for t in fig_syn.data if t.type == "scatter"]
assert len(heatmaps) >= 1, "synergy 'both' should have heatmap"
assert len(scatters) >= 1, "synergy 'both' should have dendrogram scatter"
print(f"  ✓ plot_synergy(kind='both') OK  |  heatmap={len(heatmaps)}, dendrogram traces={len(scatters)}")

fig_syn_m = report.plot_synergy(kind="matrix", sample_size=300)
print(f"  ✓ plot_synergy(kind='matrix') OK")

fig_syn_d = report.plot_synergy(kind="dendrogram", sample_size=300)
print(f"  ✓ plot_synergy(kind='dendrogram') OK")


[7] Synergy analysis (interaction SHAP):


  ✓ plot_synergy(kind='both') OK  |  heatmap=1, dendrogram traces=2


  ✓ plot_synergy(kind='matrix') OK


  ✓ plot_synergy(kind='dendrogram') OK


In [16]:
# Specific pair
report.plot_pair_detail(feature_a="pclass", feature_b="sex_enc"  ,)

In [20]:
# Top 3 most synergistic pairs (auto)
report.plot_synergy_pairs(top_k=3, sample_size=200, bins=8)

[Figure({
     'data': [{'colorbar': {'len': 0.8,
                            'outlinewidth': 0,
                            'thickness': 12,
                            'tickfont': {'size': 8},
                            'tickformat': '.0%',
                            'title': {'font': {'size': 10}, 'text': 'positive rate'},
                            'x': 0.47},
               'colorscale': [[0.0, '#FF8C00'], [0.5, '#0D1B2A'], [1.0, '#80CBC4']],
               'customdata': {'bdata': 'HhBIDxIc', 'dtype': 'i1', 'shape': '2, 3'},
               'hovertemplate': ('pclass: %{x}<br>sex_enc: %{y}<' ... ': %{customdata}<extra></extra>'),
               'showscale': True,
               'text': [['36.7%', '18.8%', '13.9%'], ['93.3%', '94.4%', '50.0%']],
               'textfont': {'color': 'white', 'size': 9},
               'texttemplate': '%{text}',
               'type': 'heatmap',
               'x': [1, 2, 3],
               'xaxis': 'x',
               'y': [0, 1],
               'y

## 8. XAI: Redundancy

In [ ]:
print("\n[8] Redundancy analysis (SHAP vector correlation):")
fig_red = report.plot_redundancy(kind="all")
heatmaps = [t for t in fig_red.data if t.type == "heatmap"]
# assert len(heatmaps) >= 1, "redundancy 'both' should have heatmap"
print(f"  ✓ plot_redundancy(kind='both') OK  |  heatmap={len(heatmaps)}")


[8] Redundancy analysis (SHAP vector correlation):


  ✓ plot_redundancy(kind='both') OK  |  heatmap=1


In [19]:
# Specific pair
report.plot_pair_detail(feature_a="parch", feature_b="sibsp"  ,)

## 9. XAI: Top interactions

In [ ]:
print("\n[9] Top feature interactions:")
top_ix = report.top_interactions(top_k=10, sample_size=300)
assert list(top_ix.columns) == ["feature_a", "feature_b", "mean_abs_interaction"]
assert len(top_ix) <= 10
assert top_ix["mean_abs_interaction"].is_monotonic_decreasing
print(f"  ✓ top_interactions() OK  |  top pair: {top_ix.iloc[0]['feature_a']} ↔ {top_ix.iloc[0]['feature_b']}")
print(top_ix.head(5).to_string(index=False))


[9] Top feature interactions:
  ✓ top_interactions() OK  |  top pair: pclass ↔ sex_enc
feature_a feature_b  mean_abs_interaction
   pclass   sex_enc              0.016352
   pclass   who_enc              0.014412
   pclass      fare              0.008009
     fare   who_enc              0.005646
  sex_enc      fare              0.005470


## 10. XAI: PDP

In [ ]:
print("\n[10] Partial Dependence Plot:")
fig_pdp = report.plot_pdp("age", grid_points=20, ice_samples=50)
assert len(fig_pdp.data) >= 2, "PDP should have >= 2 traces"
print(f"  ✓ plot_pdp('age', ice_samples=50) OK  |  traces={len(fig_pdp.data)}")

fig_pdp2 = report.plot_pdp("fare", grid_points=20)
print(f"  ✓ plot_pdp('fare') OK")


[10] Partial Dependence Plot:


  ✓ plot_pdp('age', ice_samples=50) OK  |  traces=52


  ✓ plot_pdp('fare') OK


## 11. XAI: Simulate

In [ ]:
print("\n[11] Feature simulation:")
sim_age = report.simulate("age", grid=list(range(5, 75, 5)), hold="median")
assert sim_age.shape == (14, 2)
assert list(sim_age.columns) == ["age", "prediction"]
print(f"  ✓ simulate('age', grid=range(5,75,5)) OK")
print(f"  Age 5: {sim_age.iloc[0]['prediction']:.3f} → Age 70: {sim_age.iloc[-1]['prediction']:.3f}")

sim_class = report.simulate("pclass", grid=[1, 2, 3], hold="mean")
assert sim_class.shape == (3, 2)
print(f"  ✓ simulate('pclass', grid=[1,2,3]) OK")
print(f"  pclass 1={sim_class.iloc[0]['prediction']:.3f}, 2={sim_class.iloc[1]['prediction']:.3f}, 3={sim_class.iloc[2]['prediction']:.3f}")

print("\n" + "=" * 60)
print("✅ Tutorial 1 (Titanic Classification): ALL CHECKS PASSED")
print("=" * 60)


[11] Feature simulation:
  ✓ simulate('age', grid=range(5,75,5)) OK
  Age 5: 0.309 → Age 70: 0.118
  ✓ simulate('pclass', grid=[1,2,3]) OK
  pclass 1=0.301, 2=0.149, 3=0.212

✅ Tutorial 1 (Titanic Classification): ALL CHECKS PASSED
